In [ ]:
# Projeto: Assistente de Voz com Whisper + ChatGPT + gTTS
# Baseado em estudos de reconhecimento de fala, com adaptações na estrutura e implementação

from IPython.display import Audio, display, Javascript
from google.colab import output
from base64 import b64decode
import os


# BLOCO 1: CAPTURA DE ÁUDIO


JS_GRAVADOR = """
const esperar = tempo => new Promise(resolve => setTimeout(resolve, tempo))

const blobParaBase64 = blob => new Promise(resolve => {
  const leitor = new FileReader()
  leitor.onloadend = e => resolve(e.target.result)
  leitor.readAsDataURL(blob)
})

var iniciarGravacao = tempo => new Promise(async resolve => {
  const stream = await navigator.mediaDevices.getUserMedia({ audio: true })
  const recorder = new MediaRecorder(stream)
  let partes = []

  recorder.ondataavailable = e => partes.push(e.data)
  recorder.start()

  await esperar(tempo)

  recorder.onstop = async () => {
    const blob = new Blob(partes)
    const base64 = await blobParaBase64(blob)
    resolve(base64)
  }

  recorder.stop()
})
"""

def capturar_audio(segundos=5):
    """
    Captura áudio do usuário via navegador (Colab) e salva como arquivo WAV.
    """
    display(Javascript(JS_GRAVADOR))
    resultado_js = output.eval_js(f'iniciarGravacao({segundos * 1000})')

    audio_bytes = b64decode(resultado_js.split(',')[1])
    caminho_arquivo = 'audio_usuario.wav'

    with open(caminho_arquivo, 'wb') as f:
        f.write(audio_bytes)

    return f'/content/{caminho_arquivo}'


print('🎤 Gravando áudio...')
arquivo_audio = capturar_audio()

display(Audio(arquivo_audio, autoplay=False))



# BLOCO 2: TRANSCRIÇÃO (WHISPER)


!pip install git+https://github.com/openai/whisper.git -q
import whisper

modelo_whisper = whisper.load_model("small")

def transcrever_audio(caminho_audio, idioma="pt"):
    """
    Transcreve o áudio utilizando o modelo Whisper.
    """
    resultado = modelo_whisper.transcribe(caminho_audio, fp16=False, language=idioma)
    return resultado["text"]

texto_transcrito = transcrever_audio(arquivo_audio)

print("\n📝 Texto reconhecido:")
print(texto_transcrito)



# BLOCO 3: CHATGPT


!pip install openai -q
import openai

# Coloque sua API Key aqui
os.environ['OPENAI_API_KEY'] = 'SUA_API_KEY_AQUI'
openai.api_key = os.environ.get('OPENAI_API_KEY')


def gerar_resposta_chatgpt(pergunta):
    """
    Envia o texto para o ChatGPT e retorna a resposta.
    """
    resposta = openai.ChatCompletion.create(
        model="gpt-4",
        messages=[
            {"role": "system", "content": "Responda de forma clara e objetiva."},
            {"role": "user", "content": pergunta}
        ]
    )

    return resposta.choices[0].message.content


resposta_ia = gerar_resposta_chatgpt(texto_transcrito)

print("\n Resposta da IA:")
print(resposta_ia)



# BLOCO 4: TEXTO → VOZ (gTTS)

!pip install gTTS -q
from gtts import gTTS

def gerar_audio_resposta(texto, idioma="pt"):
    """
    Converte texto em áudio usando gTTS.
    """
    tts = gTTS(text=texto, lang=idioma, slow=False)
    caminho_saida = "/content/resposta_ia.mp3"
    tts.save(caminho_saida)
    return caminho_saida


audio_resposta = gerar_audio_resposta(resposta_ia)

print("\n🔊 Reproduzindo resposta em áudio...")
display(Audio(audio_resposta, autoplay=True))

🎤 Gravando áudio...


<IPython.core.display.Javascript object>